# Sistema RSA para firmas digitales

En este notebook, mostraremos el proceso para generar las llaves pública y privada, así como el proceso para generar una firma digital y su verificación correspondiente usando el sistema RSA. Empezaremos por cargar las funciones que necesitamos:

In [1]:
import sys
import importlib.util

if 'google.colab' in sys.modules:
    runtime = "Google Colab"
    if importlib.util.find_spec("cryptocalc") is None:
        print("   Installing MA2006B from GitHub\n")
        !pip install git+https://github.com/Krul-dev/MA2006B.git
else:
    runtime = "Local environment"

import cryptocalc

print(f"\n========= Notebook execution context =========")
print(f"              Runtime: {runtime}")
print(f"       Python version: {sys.version.split()[0]}")
print(f"   CryptoCalc version: {cryptocalc.__version__}\n")

from cryptocalc import (
    rsa_key_generation,
    rsa_encryption,
    rsa_decryption,
    sha256_of_sentence,
)


========= Notebook execution context =========
              Runtime: Local environment
       Python version: 3.14.3
   CryptoCalc version: 0.1.0



## Protocolo RSA para la generación de llaves

Una vez que ya hemos cargado las librerías necesarias, empezamos por generar nuestras llaves *pública* y *privada.*

In [2]:
(public_key, private_key) = rsa_key_generation(1000)
d = private_key
e = public_key[0]
n = public_key[1]

print(f"Llave privada (d): {d}\n")
print(f"Llave pública (e): {e}\n")
print(f"Módulo para las llaves (n): {n}\n")

Llave privada (d): (36810149207577746209292908705704120000932600250998932446725793304818226066567167037095599051960259843187652824885396764308440072064623799310330862102844191718588475735291957769680948419236959537808584382862710974772267783928768146492954914110631354339769563571452881763072270145014095062987786759978549393564555179927641589521029255227521578693339548495649965682917202564248382412439371667435931237575038988471881326449110342703710301576341082858513520034363974890095726654946919833794810784843005062546071734232505016570694631900149622885193494113877668604673976959565806112731045705093493331666040465, 3964416532927467878325164921195245698582001943485369479657313343551357526005081880809397020752513556404044572379731558112276074743474741890818960280079701230298318763991042260487051069071735714897392353085639731074821063819331621009512893916499781550919407558966521251736125958873642194411080294630768029764800505971067515473409139732606018595615369542351244088339375323749327

Recordemos que el valor $d$ de la llave privada se debe de mantener en secreto. En cambio los valores de $e$ y $n$ corresponden a la llave pública y son conocidos por todos los agentes involucrados, incluída Eva.

## Protocolo RSA para la firma de mensajes

Para ilustrar el algoritmo de generación de firmas digitales, supongamos que Alicia desea firmar el siguiente mensaje llano $m$:

In [3]:
m = "Hello World!"
h = sha256_of_sentence(m)

print(f"Mensaje llano (m): {m}\n")
print(f"Hash del mensaje llano (h): {h}\n")

Mensaje llano (m): Hello World!

Hash del mensaje llano (h): 57676413081093003148005107550719583540116985236696423860923466490497932824681



Para generar la firma digital $s$, Alicia debe de utilizar el mismo algoritmo que utiliza para descifrar mensajes. En otras palabras, Alicia debe calcular
$$
s = h^{d} \mod n
$$
y ya con esta información puede generar el *mensaje firmado*
$$
\text{mensaje\_firmado} = (m, s)
$$

In [4]:
s = rsa_decryption(private_key, h)
signed_message = (m, s)

print(f"El mensaje firmado es: {signed_message}\n")

El mensaje firmado es: ('Hello World!', 12063176981210790120050783596430379010513363025780578556970476778735752172183866873695191967709589071154954879449123263500655596477124061121770211442022016273335218072500091216335642661075807063928241562270809368516363759422970107515986693121485908286367115534838973885939879171324114329835636052479830096896078316678592431526911571951517896691461188176261508233589343241110326152074160560350288504004253390305427653944674477034290009861446466686651524503851455253560306244508716658208030506269911531310040640408600486279655413797115721773408520913585231542276860793049009163807109141772290910373566001)



Para verificar la firma, Beto aplica ahora el mismo algoritmo que utilizaría para encriptar mensajes. De manera más precisa, Beto calcula el Hash
$$
h = \operatorname{Hash}(m)
$$
y calcula también
$$
\tilde{h} = s^{e} \mod n
$$
Si estos dos números son iguales, entonces la firma es válida.

In [5]:
h = sha256_of_sentence(signed_message[0])
s = signed_message[1]
h_tilde = rsa_encryption(public_key, s)

if h == h_tilde:
    verificacion_firma = "la firma es válida"
else:
    verificacion_firma = "la firma es inválida"

print(f"El valor de h es: {h}\n")
print(f"El valor de h̃ es: {h_tilde}\n")
print("Por lo tanto,", verificacion_firma)

El valor de h es: 57676413081093003148005107550719583540116985236696423860923466490497932824681

El valor de h̃ es: 57676413081093003148005107550719583540116985236696423860923466490497932824681

Por lo tanto, la firma es válida
